# STAR `rz` Integration Checks

This notebook checks the Gemini Logical `star_rz` path end to end for one logical Steane block:

1. prepare a logical `|+>` state,
2. apply `gemini.logical.star_rz(theta, q)`,
3. lower through the Lanes transversal STAR rewrite and physical SQuIn emission,
4. construct the resulting `tsim.Circuit`,
5. project the state onto the `+1` eigenspace of the Steane X stabilizers, and
6. compare the accepted logical state to a one-qubit `Rz(theta)|+>` target.

The final cell reports a PASS/FAIL-style status instead of raising, so the notebook can be executed while still making any mismatch obvious.

In [1]:
import math
import warnings

import numpy as np
import stim
from kirin import rewrite
from kirin.dialects import py

from bloqade import squin, tsim
from bloqade.gemini import logical as gemini_logical
from bloqade.lanes.arch.gemini.physical import (
    get_arch_spec as get_physical_arch_spec,
)
from bloqade.lanes.logical_mvp import compile_squin_to_move
from bloqade.lanes.noise_model import generate_logical_noise_model
from bloqade.lanes.rewrite.squin2stim import RemoveReturn
from bloqade.lanes.transform import MoveToSquinLogical
from bloqade.squin.gate import stmts as gate_stmts


In [2]:
THETA = math.pi / 16

STEANE_X_STABILIZERS = (
    (0, 1, 2, 3),
    (1, 2, 4, 5),
    (2, 3, 4, 6),
)

# These are equivalent Steane logical-Z representatives.  The STAR gate
# defaults to the weight-3 line (4, 5, 6).  The measurement post-processing
# convention also uses (0, 1, 5), which differs by a Z stabilizer.
LOGICAL_Z_SUPPORT = (4, 5, 6)
LOGICAL_X_SUPPORT = (0, 1, 5)

FIDELITY_THRESHOLD = 0.999

## Kernels

The STAR test kernel uses `squin.h` to prepare the logical `|+>` state from the default logical allocation, then applies one `star_rz`.
The target kernel is the corresponding one-qubit `H; Rz(theta)` circuit.

In [3]:
@gemini_logical.kernel(aggressive_unroll=True, verify=False)
def star_on_plus_kernel():
    reg = squin.qalloc(1)
    squin.h(reg[0])
    gemini_logical.star_rz(THETA, reg[0])


@gemini_logical.kernel(aggressive_unroll=True, verify=False)
def logical_plus_kernel():
    reg = squin.qalloc(1)
    squin.h(reg[0])


@squin.kernel(typeinfer=True)
def one_qubit_target_kernel():
    reg = squin.qalloc(1)
    squin.h(reg[0])
    squin.rz(THETA, reg[0])
    return reg

## Compilation Helpers

`star_on_plus_kernel` goes through the same STAR path used by the demo: Gemini Logical -> Move with transversal STAR rewrite -> physical SQuIn -> `tsim.Circuit`.

SQuIn `U3` parameters are in turns, while the current `move.LocalRz` lowering passes a radian-valued STAR angle through directly.  The notebook therefore keeps both versions around: the current emitted circuit, and a diagnostic circuit with the STAR `U3` angle divided by `tau`.


In [4]:
def remove_return(method):
    out = method.similar()
    rewrite.Walk(RemoveReturn()).rewrite(out.code)
    return out


def squin_to_tsim_circuit(method):
    return tsim.Circuit(remove_return(method))


def constant_float(value) -> float | None:
    owner = value.owner
    if isinstance(owner, py.Constant):
        return float(owner.value.unwrap())
    return None


def convert_star_u3_radian_angles_to_turns(method):
    out = method.similar()
    for stmt in list(out.callable_region.walk()):
        if not isinstance(stmt, gate_stmts.U3):
            continue
        if constant_float(stmt.theta) != 0.0 or constant_float(stmt.lam) != 0.0:
            continue

        zero = py.Constant(0.0)
        tau = py.Constant(math.tau)
        scaled_angle = py.Div(stmt.phi, tau.result)
        zero.insert_before(stmt)
        tau.insert_before(stmt)
        scaled_angle.insert_before(stmt)
        stmt.replace_by(
            gate_stmts.U3(
                stmt.theta,
                zero.result,
                scaled_angle.result,
                stmt.qubits,
            )
        )
    return out


def gemini_logical_to_physical_squin(method):
    physical_move = compile_squin_to_move(
        method,
        transversal_rewrite=True,
        no_raise=False,
        insert_return_moves=False,
    )
    physical_squin = MoveToSquinLogical(
        arch_spec=get_physical_arch_spec(),
        noise_model=generate_logical_noise_model(),
        add_noise=False,
    ).emit(physical_move, no_raise=False)
    return physical_move, physical_squin


star_move_kernel, star_physical_squin_current = gemini_logical_to_physical_squin(
    star_on_plus_kernel
)
star_physical_squin = convert_star_u3_radian_angles_to_turns(
    star_physical_squin_current
)
_, plus_physical_squin = gemini_logical_to_physical_squin(logical_plus_kernel)

star_tsim_circuit_current = squin_to_tsim_circuit(star_physical_squin_current)
star_tsim_circuit = squin_to_tsim_circuit(star_physical_squin)
plus_tsim_circuit = squin_to_tsim_circuit(plus_physical_squin)
target_tsim_circuit = squin_to_tsim_circuit(one_qubit_target_kernel)

print(f"Current emitted STAR tsim qubits: {star_tsim_circuit_current.num_qubits}")
print(f"Current emitted STAR is Clifford: {star_tsim_circuit_current.is_clifford}")
print(f"Current emitted STAR measurements: {star_tsim_circuit_current.num_measurements}")
print()
print(star_tsim_circuit_current.stim_circuit)
print()
print("Diagnostic STAR circuit with U3 angle converted from radians to turns:")
print(star_tsim_circuit.stim_circuit)


Current emitted STAR tsim qubits: 7
Current emitted STAR is Clifford: False
Current emitted STAR measurements: 0

SQRT_Y_DAG 0 1 2 3 4 5
CZ 1 2 3 4 5 6
SQRT_Y 6
CZ 0 3 2 5 4 6
SQRT_Y 2 3 4 5 6
CZ 0 1 2 3 4 5
SQRT_Y 1 2 4
X 3
Z 1 5
S 0 1 2 3 4 5 6
SQRT_X 0 1 2 3 4 5 6
S 0 1 2 3 4 5 6
I[U3(theta=0.0*pi, phi=-1.73053616*pi, lambda=0.0*pi)] 4 5 6

Diagnostic STAR circuit with U3 angle converted from radians to turns:
SQRT_Y_DAG 0 1 2 3 4 5
CZ 1 2 3 4 5 6
SQRT_Y 6
CZ 0 3 2 5 4 6
SQRT_Y 2 3 4 5 6
CZ 0 1 2 3 4 5
SQRT_Y 1 2 4
X 3
Z 1 5
S 0 1 2 3 4 5 6
SQRT_X 0 1 2 3 4 5 6
S 0 1 2 3 4 5 6
I[U3(theta=0.0*pi, phi=0.0*pi, lambda=-0.27542338*pi)] 4 5 6


## State And Pauli Helpers

The `tsim.Circuit.to_matrix()` calls below are only used to obtain exact state vectors for the compiled circuits.  Then we apply the stabilizer postselection and expectation-value comparison ourselves.

Important convention: `tsim.Circuit.to_matrix()` stores qubit 0 as the high-order state-vector bit, so the Pauli helper maps physical qubit index `q` to bit index `num_qubits - 1 - q`.


In [5]:
def initial_state(num_qubits: int) -> np.ndarray:
    state = np.zeros(2**num_qubits, dtype=complex)
    state[0] = 1.0
    return state


def normalized(state: np.ndarray) -> tuple[np.ndarray, float]:
    norm = np.linalg.norm(state)
    if norm == 0:
        raise ValueError("cannot normalize the zero vector")
    return state / norm, float(norm * norm)


def state_from_tsim(circuit: tsim.Circuit) -> np.ndarray:
    with warnings.catch_warnings():
        # tsim may warn while constructing a full non-Clifford matrix even
        # when the input-state column we use is finite.
        warnings.simplefilter("ignore", RuntimeWarning)
        matrix = circuit.to_matrix()
        state = matrix @ initial_state(circuit.num_qubits)
    state, _ = normalized(state)
    return state


def apply_pauli(state: np.ndarray, paulis: dict[int, str]) -> np.ndarray:
    num_qubits = int(math.log2(len(state)))
    out = np.zeros_like(state)
    for basis_state, amplitude in enumerate(state):
        dest = basis_state
        coeff = 1.0 + 0.0j
        for qubit_index, pauli in paulis.items():
            bit_index = num_qubits - 1 - qubit_index
            bit = (basis_state >> bit_index) & 1
            if pauli == "X":
                dest ^= 1 << bit_index
            elif pauli == "Y":
                dest ^= 1 << bit_index
                coeff *= 1j if bit == 0 else -1j
            elif pauli == "Z":
                coeff *= -1 if bit else 1
            else:
                raise ValueError(f"unsupported Pauli {pauli!r}")
        out[dest] += coeff * amplitude
    return out


def expectation(state: np.ndarray, paulis: dict[int, str]) -> complex:
    return np.vdot(state, apply_pauli(state, paulis))


def apply_single_qubit_unitary(
    state: np.ndarray, qubit_index: int, unitary: np.ndarray
) -> np.ndarray:
    num_qubits = int(math.log2(len(state)))
    bit_index = num_qubits - 1 - qubit_index
    out = state.copy()
    for basis_state in range(len(state)):
        if ((basis_state >> bit_index) & 1) != 0:
            continue
        paired_state = basis_state | (1 << bit_index)
        amp_0 = state[basis_state]
        amp_1 = state[paired_state]
        out[basis_state] = unitary[0, 0] * amp_0 + unitary[0, 1] * amp_1
        out[paired_state] = unitary[1, 0] * amp_0 + unitary[1, 1] * amp_1
    return out


def basis_change_pauli_expectation(
    state: np.ndarray, paulis: dict[int, str]
) -> complex:
    h = np.array([[1, 1], [1, -1]], dtype=complex) / math.sqrt(2)
    s_dag = np.array([[1, 0], [0, -1j]], dtype=complex)
    rotated = state.copy()
    for qubit_index, pauli in paulis.items():
        if pauli == "X":
            rotated = apply_single_qubit_unitary(rotated, qubit_index, h)
        elif pauli == "Y":
            rotated = apply_single_qubit_unitary(rotated, qubit_index, s_dag)
            rotated = apply_single_qubit_unitary(rotated, qubit_index, h)
        elif pauli == "Z":
            pass
        else:
            raise ValueError(f"unsupported Pauli {pauli!r}")
    return expectation(rotated, {qubit_index: "Z" for qubit_index in paulis})


def project_x_stabilizer_plus(
    state: np.ndarray, support: tuple[int, ...]
) -> np.ndarray:
    x_string = {qubit_index: "X" for qubit_index in support}
    return 0.5 * (state + apply_pauli(state, x_string))


def postselect_x_stabilizers(state: np.ndarray) -> tuple[np.ndarray, float]:
    accepted = state.copy()
    for support in STEANE_X_STABILIZERS:
        accepted = project_x_stabilizer_plus(accepted, support)
    return normalized(accepted)


## Exact Postselected Comparison

This computes the state prepared by the compiled STAR `tsim` circuit, postselects the X stabilizers, and compares logical Bloch-vector expectation values to the single-qubit target.

`Y_L` is checked two ways: directly as a Pauli string and by applying the corresponding measurement-basis changes before evaluating a Z-basis parity.


In [6]:
def logical_bloch_after_postselection(circuit: tsim.Circuit):
    state = state_from_tsim(circuit)
    accepted_state, acceptance_probability = postselect_x_stabilizers(state)

    logical_y = {
        0: "X",
        1: "X",
        4: "Z",
        5: "Y",
        6: "Z",
    }
    logical_bloch = {
        "X": expectation(
            accepted_state, {qubit_index: "X" for qubit_index in LOGICAL_X_SUPPORT}
        ),
        "Y": expectation(accepted_state, logical_y),
        "Z": expectation(
            accepted_state, {qubit_index: "Z" for qubit_index in LOGICAL_Z_SUPPORT}
        ),
    }
    basis_change_y = basis_change_pauli_expectation(accepted_state, logical_y)
    stabilizer_expectations = [
        expectation(
            accepted_state, {qubit_index: "X" for qubit_index in support}
        )
        for support in STEANE_X_STABILIZERS
    ]
    return (
        accepted_state,
        acceptance_probability,
        logical_bloch,
        basis_change_y,
        stabilizer_expectations,
    )


target_state = state_from_tsim(target_tsim_circuit)
target_bloch = {
    "X": np.vdot(target_state, np.array([target_state[1], target_state[0]])),
    "Y": np.vdot(
        target_state,
        np.array([-1j * target_state[1], 1j * target_state[0]]),
    ),
    "Z": np.vdot(target_state, np.array([target_state[0], -target_state[1]])),
}

(
    accepted_state_current,
    acceptance_probability_current,
    logical_bloch_current,
    basis_change_y_current,
    stabilizer_expectations_current,
) = logical_bloch_after_postselection(star_tsim_circuit_current)

(
    accepted_state,
    acceptance_probability,
    logical_bloch,
    basis_change_y,
    stabilizer_expectations,
) = logical_bloch_after_postselection(star_tsim_circuit)

target_vector = np.array(
    [target_bloch[axis].real for axis in ("X", "Y", "Z")]
)
logical_vector_current = np.array(
    [logical_bloch_current[axis].real for axis in ("X", "Y", "Z")]
)
logical_vector = np.array(
    [logical_bloch[axis].real for axis in ("X", "Y", "Z")]
)
bloch_fidelity_current = (
    1.0 + float(np.dot(target_vector, logical_vector_current))
) / 2.0
bloch_fidelity = (1.0 + float(np.dot(target_vector, logical_vector))) / 2.0

print(f"theta = {THETA:.12f} radians")
print(
    "current emitted STAR postselection probability = "
    f"{acceptance_probability_current:.12f}"
)
print(
    "diagnostic turns-corrected STAR postselection probability = "
    f"{acceptance_probability:.12f}"
)
print()
print("X stabilizers after postselection:")
for support, value in zip(STEANE_X_STABILIZERS, stabilizer_expectations):
    print(f"  X{support}: {value.real:+.12f}{value.imag:+.2e}j")
print()
print("Single-qubit target Bloch vector:")
for axis in ("X", "Y", "Z"):
    value = target_bloch[axis]
    print(f"  <{axis}> = {value.real:+.12f}{value.imag:+.2e}j")
print()
print("Current emitted logical Bloch vector:")
for axis in ("X", "Y", "Z"):
    value = logical_bloch_current[axis]
    print(f"  <{axis}_L> = {value.real:+.12f}{value.imag:+.2e}j")
print(f"  <Y_L> via basis change = {basis_change_y_current.real:+.12f}")
print(f"  fidelity = {bloch_fidelity_current:.12f}")
print()
print("Turns-corrected diagnostic logical Bloch vector:")
for axis in ("X", "Y", "Z"):
    value = logical_bloch[axis]
    print(f"  <{axis}_L> = {value.real:+.12f}{value.imag:+.2e}j")
print(f"  <Y_L> via basis change = {basis_change_y.real:+.12f}")
print()
print(f"Bloch-state fidelity estimate = {bloch_fidelity:.12f}")
print(
    "STATUS:",
    "PASS" if bloch_fidelity >= FIDELITY_THRESHOLD else "FAIL",
    f"(threshold {FIDELITY_THRESHOLD})",
)


theta = 0.196349540849 radians
current emitted STAR postselection probability = 0.579253637500
diagnostic turns-corrected STAR postselection probability = 0.565352000402

X stabilizers after postselection:
  X(0, 1, 2, 3): +1.000000000000+0.00e+00j
  X(1, 2, 4, 5): +1.000000000000+0.00e+00j
  X(2, 3, 4, 6): +1.000000000000+0.00e+00j

Single-qubit target Bloch vector:
  <X> = +0.980785280403+2.82e-18j
  <Y> = +0.195090322016+1.02e-17j
  <Z> = -0.000000000000+0.00e+00j

Current emitted logical Bloch vector:
  <X_L> = +0.983419207542+0.00e+00j
  <Y_L> = -0.181346801013+0.00e+00j
  <Z_L> = -0.000000000000+0.00e+00j
  <Y_L> via basis change = -0.181346801013
  fidelity = 0.964572038708

Turns-corrected diagnostic logical Bloch vector:
  <X_L> = +0.980785281957+0.00e+00j
  <Y_L> = +0.195090314205+0.00e+00j
  <Z_L> = -0.000000000000+0.00e+00j
  <Y_L> via basis change = +0.195090314205

Bloch-state fidelity estimate = 1.000000000000
STATUS: PASS (threshold 0.999)


## Sampling-Based Fidelity Estimate

The exact-state check above projects the state vector onto the `+1` eigenspace of the three X stabilizers.  The sampling check below does the measurement version of the same thing: it appends commuting `MPP` measurements for the three X stabilizers and one logical Pauli observable, samples those `tsim` circuits, postselects shots whose stabilizer outcomes are all `+1`, and estimates the logical Bloch vector from the accepted shots.

This uses `MPP` instead of simple terminal all-X/all-Y/all-Z basis kernels because the logical observable and the X-stabilizer postselection must be sampled jointly.  For example, `Y_L` commutes with the X stabilizers globally, but not qubit-by-qubit in a single product basis.


In [8]:
SAMPLING_SHOTS = 50_000


def stim_target(pauli: str, qubit_index: int):
    if pauli == "X":
        return stim.target_x(qubit_index)
    if pauli == "Y":
        return stim.target_y(qubit_index)
    if pauli == "Z":
        return stim.target_z(qubit_index)
    raise ValueError(f"unsupported Pauli {pauli!r}")


def append_mpp_product(circuit: stim.Circuit, paulis: dict[int, str]) -> None:
    targets = []
    for index, (qubit_index, pauli) in enumerate(sorted(paulis.items())):
        if index:
            targets.append(stim.target_combiner())
        targets.append(stim_target(pauli, qubit_index))
    circuit.append("MPP", targets)


def mpp_sampler_for_observable(
    base_circuit: tsim.Circuit, observable_paulis: dict[int, str]
) -> tsim.Circuit:
    stim_circuit = base_circuit.stim_circuit.copy()
    for support in STEANE_X_STABILIZERS:
        append_mpp_product(
            stim_circuit, {qubit_index: "X" for qubit_index in support}
        )
    append_mpp_product(stim_circuit, observable_paulis)
    return tsim.Circuit.from_stim_program(stim_circuit)


def estimate_postselected_observable(
    base_circuit: tsim.Circuit,
    observable_paulis: dict[int, str],
    *,
    shots: int,
    seed: int,
):
    measurement_circuit = mpp_sampler_for_observable(base_circuit, observable_paulis)
    samples = measurement_circuit.compile_sampler(seed=seed).sample(shots)

    stabilizer_samples = samples[:, : len(STEANE_X_STABILIZERS)]
    observable_samples = samples[:, len(STEANE_X_STABILIZERS)]

    # Stim/tsim measurement bit False means the +1 eigenvalue; True means -1.
    accepted = ~np.any(stabilizer_samples, axis=1)
    accepted_observable_samples = observable_samples[accepted]
    eigenvalues = np.where(accepted_observable_samples, -1.0, 1.0)

    expectation_estimate = float(np.mean(eigenvalues))
    standard_error = float(
        np.sqrt(max(0.0, 1.0 - expectation_estimate**2) / len(eigenvalues))
    )
    return {
        "expectation": expectation_estimate,
        "standard_error": standard_error,
        "accepted": int(np.sum(accepted)),
        "shots": int(shots),
        "acceptance_probability": float(np.mean(accepted)),
    }


LOGICAL_PAULIS = {
    "X": {qubit_index: "X" for qubit_index in LOGICAL_X_SUPPORT},
    "Y": {
        0: "X",
        1: "X",
        4: "Z",
        5: "Y",
        6: "Z",
    },
    "Z": {qubit_index: "Z" for qubit_index in LOGICAL_Z_SUPPORT},
}


def sample_logical_bloch(base_circuit: tsim.Circuit, *, shots: int, seed: int):
    estimates = {}
    for offset, axis in enumerate(("X", "Y", "Z")):
        estimates[axis] = estimate_postselected_observable(
            base_circuit,
            LOGICAL_PAULIS[axis],
            shots=shots,
            seed=seed + offset,
        )
    vector = np.array([estimates[axis]["expectation"] for axis in ("X", "Y", "Z")])
    fidelity = (1.0 + float(np.dot(target_vector, vector))) / 2.0
    return estimates, vector, fidelity


sample_estimates_current, sample_vector_current, sample_fidelity_current = (
    sample_logical_bloch(
        star_tsim_circuit_current, shots=SAMPLING_SHOTS, seed=20260514
    )
)
sample_estimates, sample_vector, sample_fidelity = sample_logical_bloch(
    star_tsim_circuit, shots=SAMPLING_SHOTS, seed=20260524
)

print(f"shots per observable = {SAMPLING_SHOTS}")
print()
print("Current emitted STAR sampling estimates:")
for axis in ("X", "Y", "Z"):
    result = sample_estimates_current[axis]
    print(
        f"  <{axis}_L> = {result['expectation']:+.6f} "
        f"+/- {result['standard_error']:.6f} "
        f"(accepted {result['accepted']}/{result['shots']})"
    )
print(f"  sampled fidelity estimate = {sample_fidelity_current:.6f}")
print()
print("Unit-corrected diagnostic STAR sampling estimates:")
for axis in ("X", "Y", "Z"):
    result = sample_estimates[axis]
    print(
        f"  <{axis}_L> = {result['expectation']:+.6f} "
        f"+/- {result['standard_error']:.6f} "
        f"(accepted {result['accepted']}/{result['shots']})"
    )
print(f"  sampled fidelity estimate = {sample_fidelity:.6f}")
print()
print("Exact unit-corrected Bloch vector:")
for axis in ("X", "Y", "Z"):
    print(f"  <{axis}_L> = {logical_bloch[axis].real:+.6f}")
print(f"  exact fidelity = {bloch_fidelity:.6f}")


shots per observable = 50000

Current emitted STAR sampling estimates:
  <X_L> = +0.984312 +/- 0.001037 (accepted 28940/50000)
  <Y_L> = -0.180900 +/- 0.005785 (accepted 28900/50000)
  <Z_L> = -0.004606 +/- 0.005863 (accepted 29092/50000)
  sampled fidelity estimate = 0.965054

Unit-corrected diagnostic STAR sampling estimates:
  <X_L> = +0.981298 +/- 0.001146 (accepted 28232/50000)
  <Y_L> = +0.187595 +/- 0.005841 (accepted 28279/50000)
  <Z_L> = +0.004205 +/- 0.005944 (accepted 28299/50000)
  sampled fidelity estimate = 0.999520

Exact unit-corrected Bloch vector:
  <X_L> = +0.980785
  <Y_L> = +0.195090
  <Z_L> = -0.000000
  exact fidelity = 1.000000
